# Experiment — RAG sobre el Sistema Penal Acusatorio (México)

Un **segundo corpus** para probar el mismo pipeline en otro dominio y otro idioma:
8 documentos jurídicos (dos códigos — CNPP y Código Penal Federal — y seis obras
doctrinales) sobre el sistema penal acusatorio mexicano.

Esto es exactamente para lo que sirve el parámetro `table=` que agregamos a
`ram_rag`: este corpus vive en **su propia tabla** (`penal_chunks`), separado por
completo de los vectores del *Root Apical Meristem* (`ram_chunks`). Un embedder,
dos corpus, dos tablas.

**Preguntas y respuestas de referencia:** en `ingestion/penal_qa.json`, en español
(mismo idioma del corpus, para que el embedder recupere sobre el mismo vocabulario).
Cada respuesta está fundamentada en un artículo concreto.

**Antes de correr:** `./tunnel.sh` (Postgres + TEI). El embedder debe ser
**multilingüe** — ver la nota en el paso de *embed*.

In [ ]:
%matplotlib inline
import sys, json
from pathlib import Path

# Mismo patrón que el otro notebook: subir hasta la carpeta que contiene `shared/`.
current_dir = Path.cwd()
labs_folder = next(d for d in [current_dir, *current_dir.parents] if (d / "shared").exists())
for import_dir in (str(labs_folder), str(labs_folder / "ingestion")):
    if import_dir not in sys.path:
        sys.path.insert(0, import_dir)

import ram_rag
from shared.embedder import get_embedder

# Este corpus tiene su propia carpeta y su propia tabla.
PENAL_DIR = labs_folder / "ingestion" / "out" / "Sistema Penal Acusatorio"
TABLE = "penal_chunks"

qa_items = json.loads((labs_folder / "ingestion" / "penal_qa.json").read_text())["questions"]
in_corpus_count = sum(item["in_corpus"] for item in qa_items)
control_count = sum(not item["in_corpus"] for item in qa_items)
print("corpus:", PENAL_DIR.name)
print("preguntas de muestra:", len(qa_items),
      f"({in_corpus_count} en corpus, {control_count} control)")

## 1. Cargar el corpus

`ram_rag.load_markdown()` está fijado al corpus de artículos (RAM), así que aquí
cargamos **este** corpus a mano. Todo lo que sigue —`chunk_words`, `embed_chunks`,
`store_chunks`, `retrieve`— es agnóstico al corpus y se reutiliza tal cual.

In [ ]:
documents = [(path.stem, path.read_text()) for path in sorted(PENAL_DIR.glob("*.md"))]
for name, text in documents:
    print(f"  {len(text):>9,} chars   {name}")
print(f"\n{len(documents)} documentos")

## 2. El pipeline, paso a paso

**chunk → embed → store**, igual que en el otro notebook, pero guardando en
`penal_chunks` mediante `table=`.

In [ ]:
# 1. chunk — el mismo chunker de ventana fija que usamos para los papers
chunks = [(source, chunk)
          for source, text in documents
          for chunk in ram_rag.chunk_words(text)]
print(f"1. chunk : {len(chunks)} chunks de {len(documents)} documentos")

# 2. embed — DEBE ser un modelo multilingüe (el corpus está en español).
#    'tei' aquí sirve Qwen3-Embedding (multilingüe); minilm / bge-small-en son
#    solo-inglés y recuperarían mal sobre este corpus.
embedder = get_embedder("tei")
vectors = ram_rag.embed_chunks(embedder, chunks)
print(f"2. embed : {len(vectors)} vectores, {embedder.dim}d, via {embedder.name}")

# 3. store — en una tabla SEPARADA, etiquetada con el nombre del embedder
ram_rag.store_chunks(chunks, vectors, embedder.dim, embedder.name, table=TABLE)
print(f"3. store : cargado en la tabla '{TABLE}'")

## 3. Preguntar y comparar contra la respuesta de referencia

Para cada pregunta recuperamos los `k` chunks más cercanos de `penal_chunks` y los
comparamos con la respuesta de referencia (fundamentada en un artículo). Fíjate si
el chunk recuperado proviene del documento esperado y contiene la respuesta.

In [ ]:
def show_retrieval(qa_item, top_k=3):
    retrieved = ram_rag.retrieve(embedder, qa_item["question"], top_k, table=TABLE)
    control_tag = "" if qa_item["in_corpus"] else "   [CONTROL · fuera de corpus]"
    reference_answer = qa_item["answer"]
    print("─" * 100)
    print("Pregunta :", qa_item["question"], control_tag)
    print("Referencia:", reference_answer[:200] + ("..." if len(reference_answer) > 200 else ""))
    print("Esperado :", qa_item["source"])
    print("Recuperado:")
    for source, text, distance in retrieved:
        snippet = " ".join(text.split())[:120]
        print(f"   {distance:.3f}  [{source}]  {snippet}...")
    print()

for qa_item in qa_items:
    show_retrieval(qa_item)

## 4. Una señal cuantitativa: ¿aparece el documento esperado?

Para las preguntas *en corpus*, cada respuesta de referencia registra el documento
del que proviene (`doc`). Un chequeo simple de recuperación: ¿aparece ese documento
entre los `k` primeros resultados? (`hit@k` a nivel de documento.)

In [ ]:
top_k = 3
in_corpus_items = [qa_item for qa_item in qa_items if qa_item["in_corpus"]]
expected_doc_hits = 0
for qa_item in in_corpus_items:
    retrieved_docs = {source for source, _, _ in
                      ram_rag.retrieve(embedder, qa_item["question"], top_k, table=TABLE)}
    expected_doc_hits += qa_item["doc"] in retrieved_docs
print(f"hit@{top_k} (documento esperado): {expected_doc_hits}/{len(in_corpus_items)} "
      f"= {expected_doc_hits / len(in_corpus_items):.0%}")

## Notas y siguientes pasos

- **El control fuera de corpus** (capital de Francia) debería devolver distancias
  altas: el RAG ingenuo *siempre* regresa sus chunks más cercanos, aunque sean
  irrelevantes. Nunca dice "no sé" — la misma brecha que motiva labs posteriores.
- **Embedder multilingüe:** con `minilm` o `bge-small-en` (solo-inglés) las
  distancias se disparan y la recuperación empeora. Buen experimento A/B para este
  corpus: `tei` (multilingüe) vs. un modelo solo-inglés.
- **Ruido de encabezados:** estos documentos repiten encabezados de página
  ("CÓDIGO NACIONAL... CÁMARA DE DIPUTADOS...", "X de 168"). Igual que el
  *reference-noise* de los papers, ese boilerplate contamina la recuperación —
  candidato natural a una función de limpieza específica de este corpus.